### **Student Information**
GitHub ID:joy930319-bot

Kaggle name:tsaichiaoyu

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# Project Report

## 1. Model Development

**Task:** classify 4,800 short Reddit-style posts into six emotions (anger, disgust, fear, joy, sadness, surprise). 3,360 posts are labelled for training; 1,440 are the test set. The metric is macro F1, so every class counts equally.

**Final result:** an average of fine-tuned **roberta-large** and **roberta-base** models. Best public score **0.5996** (sub9); out-of-fold macro F1 of the 5-fold version **0.602** (sub14). The two selected submissions are sub9 and sub14.

---

### 1.1 Preprocessing Steps

* **Merging the files:** `final_posts.json` (post_id, text) was joined with `data_identification.csv` (train/test) and `emotion.csv` (label). Every training post has a label, and the 1,440 test ids match `sampleSubmission.csv` exactly.
* **Looking at the data first:**
  * The classes are imbalanced: joy 875, anger 674, sadness 628, disgust 454, surprise 441, fear 288. Because macro F1 weighs all six equally, the rare classes (fear) matter as much as joy.
  * Posts are short: median 13 words, 75% have 19 words or fewer, but the longest has 508 words.
* **Text cleaning:** almost none. The transformer tokenizers handle casing, punctuation and emojis themselves, and removing them would throw away emotional signal ("sooo", "!!!", 😭).
* **Length:** posts were truncated to 128 tokens, which covers nearly every post; each batch is padded only to its own longest post to save time.
* **Validation:** at first a stratified 85/15 split (2,856 / 504 posts). Later a **stratified 5-fold cross-validation**, so that every one of the 3,360 posts is scored once (out-of-fold). This turned out to be important, see 1.3.
* **Submission format:** the Kaggle checker needs the column to be named `id`, not `post_id` as in `sampleSubmission.csv`; my first upload failed because of this.

### 1.2 Feature Engineering Steps

* **Baseline, TF-IDF:** word unigrams + bigrams and character n-grams (2 to 5, `char_wb`), `sublinear_tf`, `min_df=2`. Character n-grams were added because the posts are informal (typos, slang, stretched words).
* **Transformer features:** for the main models there is no hand-made feature. The pre-trained transformer turns each post into contextual embeddings and is fine-tuned end to end, so the "features" are learned for this task. This is why TF-IDF stopped at 0.49 while fine-tuned models reached about 0.59: TF-IDF only counts words, and it cannot tell that "I can't believe it" may be surprise or disgust depending on context.
* **Class weights:** each class gets the weight `n_samples / (n_classes × n_class_samples)` in the cross-entropy loss, so mistakes on fear count more than mistakes on joy.
* **Ensembling as a feature-level step:** the final prediction averages the softmax probabilities of several models, so models that make different mistakes correct each other.

### 1.3 Explanation of Your Model

**Training recipe (same for every transformer):** AdamW (weight decay 0.01), 10% linear warm-up then linear decay, gradient clipping at 1.0, batch size 16, class-weighted cross-entropy. Learning rate 2e-5 for base models and 1e-5 for large models. On the Kaggle GPU (Tesla T4) I also used fp16 mixed precision.

**How the model developed:**

| Submission | Model | Validation | Public |
|---|---|---|---|
| sub1 | TF-IDF + LinearSVC (5-fold CV) | 0.499 | 0.4873 |
| sub2 | distilroberta-base, laptop CPU | 0.606 | 0.5681 |
| sub3 | roberta-base, laptop CPU | 0.628 | 0.5865 |
| sub4 | roberta-base on all 3,360 posts, 2 epochs | none | 0.5739 |
| sub7 | roberta-base ×3 seeds + roberta-large | 0.635 | 0.5928 |
| sub8 | roberta-large ×3 seeds | 0.637 | 0.5929 |
| **sub9** | **roberta-large ×3 + roberta-base ×3** | 0.635 | **0.5996** |
| sub10 | roberta-large ×5 + roberta-base ×3 | 0.637 | 0.5967 |
| sub11 | roberta-large ×5 | 0.637 | 0.5855 |
| **sub14** | **5-fold roberta-large + roberta-base** | **OOF 0.602** | 0.5965 |

(Validation for sub2 to sub11 is the 504-post split; sub14 uses all 3,360 posts out-of-fold.)

**What I learned from the experiments:**
1. **Fine-tuning beats TF-IDF by a lot** (0.49 to about 0.59 on the public board).
2. **A bigger model alone did not help.** roberta-large (0.626) and deberta-v3-base (0.586) were not better than roberta-base on the 504-post split, and in 5-fold CV roberta-large (OOF 0.591) and roberta-base (0.593) were equal.
3. **The 504-post validation set is noisy.** Training roberta-base three times with different seeds gave 0.617, 0.604 and 0.612, so a 0.01 difference there is only about 5 posts. The laptop's 0.628 was partly a lucky seed. This is why I switched to 5-fold CV and chose models by the out-of-fold score on 3,360 posts instead of by the public leaderboard (about 430 posts).
4. **Mixing different models helps consistently.** Averaging roberta-large and roberta-base gave +0.01 OOF over either alone (0.602), and on the public board adding roberta-base raised the score both times I tried (sub8 to sub9, sub11 to sub10).
5. **Things that did not work:** ELECTRA-large (two of five folds collapsed, OOF 0.361), deberta-v3-large (0.560 on fold 1, only +0.003 in the ensemble), and per-class probability weights (honest check 0.602 to 0.604, within noise). I did not use them.

**Error analysis (final ensemble, out-of-fold):** accuracy 0.623, macro F1 0.602. Joy is the easiest class (F1 0.759) and disgust the hardest (0.432). The most confused pair is **anger and disgust (199 posts in both directions)**: about a quarter of disgust posts are predicted as anger. Both express disapproval, and in a one-line post the difference is often not visible, so this looks like a limit of the labels rather than of the model. That explains why larger models gave little improvement.

**Why I selected sub9 and sub14:** sub14 uses all the training data and was chosen by the most reliable score (OOF on 3,360 posts); sub9 has the highest public score and uses the same kind of large + base mix. Both are averages of several models, which should make them more stable on the private leaderboard.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude (an AI assistant) during the whole competition. My main prompts and what I did with the suggestions:

| My prompt (summarized) | AI suggestion | What I decided and why |
|---|---|---|
| How should I start? Give me a baseline. | TF-IDF (word + character n-grams) with LinearSVC, class weights for the imbalance, 5-fold CV. | **Used** as the first model: fast, and TF-IDF + SVM was the best model in the Master notebook. It scored 0.487, below the 0.5 baseline. |
| TF-IDF is below 0.5, how can I improve? | Fine-tune a pre-trained transformer (distilroberta-base), and do not use external emotion datasets or models already trained on emotion data. | **Used.** Transformers understand meaning, which matters for short posts. I followed the warning about external data: the posts look like public Reddit data, so an outside emotion dataset could contain the test posts, which would be cheating. I also checked that the models' licenses (Apache 2.0 / MIT) are allowed by the competition rules. |
| Training on my laptop took 2 hours. | The CPU build of PyTorch cannot use my NVIDIA card; use Kaggle's free GPU instead; keep the laptop from sleeping. | **Used** Kaggle's GPU (epochs went from about 4 to 10 minutes to under 1 minute). |
| Try roberta-base / roberta-large / deberta. | Keep the best epoch on the validation set; use a smaller learning rate for large models; load deberta in fp32 for mixed precision. | **Used.** roberta-base improved the public score to 0.5865; the larger models did not help on their own. |
| My scores go up and down, which one is really better? | Train with different seeds and average them (ensemble), and use 5-fold cross-validation so the score is measured on all 3,360 posts. | **Used.** The seed experiment showed how noisy the 504-post split is, and 5-fold CV gave the final choice (sub14, OOF 0.602). |
| Can the score go higher? | Add a different model family (ELECTRA-large, deberta-v3-large) or tune class weights for macro F1. | **Tried, not used.** Each change was checked with an honest out-of-fold test first and none improved beyond noise, so I kept the simpler ensemble. |
| Why can't my model go past 0.60? | Look at the confusion matrix. | **Used** for the error analysis in 1.3: anger vs. disgust is the main source of errors. |

What I learned from working with the AI: its suggestions were useful starting points, but I only kept a change when my own validation numbers supported it, and I checked the competition rules myself before using any pre-trained model.

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
### Add the code related to the preprocessing steps in cells inside this section

# Load the competition files and look at their structure before doing anything else
import json
import pandas as pd

DATA_DIR = "kaggle/2026-dm-lab-2-private-competition"

with open(f"{DATA_DIR}/final_posts.json", encoding="utf-8") as f:
    posts_raw = json.load(f)

emotion = pd.read_csv(f"{DATA_DIR}/emotion.csv")
data_id = pd.read_csv(f"{DATA_DIR}/data_identification.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sampleSubmission.csv")

# The JSON's top level could be a list or a dict, so print its type and first entry
print(type(posts_raw), len(posts_raw))
first = posts_raw[0] if isinstance(posts_raw, list) else list(posts_raw.items())[:1]
print(json.dumps(first, ensure_ascii=False, indent=2)[:800])

for name, df in [("emotion", emotion), ("data_identification", data_id), ("sampleSubmission", sample_sub)]:
    print(f"\n===== {name} {df.shape}")
    print(df.head(3))

<class 'list'> 4800
{
  "post_id": "f8bv5zh",
  "text": "Thought it looked weird as well."
}

===== emotion (3360, 2)
   post_id    label
0  f8bv5zh  disgust
1  q0rr1l9  disgust
2  p94rwgk  sadness

===== data_identification (4800, 2)
   post_id    set
0  f8bv5zh  train
1  q0rr1l9  train
2  p94rwgk  train

===== sampleSubmission (1440, 2)
   post_id    label
0  xyzx9jp  sadness
1  l3vzzxt  sadness
2  g680iug  sadness


In [2]:
# Merge the three files into one table, then split it into train and test
posts = pd.DataFrame(posts_raw)                      # post_id, text
df = posts.merge(data_id, on="post_id", how="left")  # add train/test split
df = df.merge(emotion, on="post_id", how="left")     # add label (missing for test)

train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)
print("train:", train_df.shape, "test:", test_df.shape)
print("train rows without label:", train_df["label"].isna().sum())

# Label balance tells us whether macro F1 will punish rare classes
print("\nlabel counts:\n", train_df["label"].value_counts())

# Text length in words, to see how short these posts are
train_df["n_words"] = train_df["text"].str.split().str.len()
print("\nwords per post:\n", train_df["n_words"].describe().round(1))

# Test set order must match sampleSubmission
print("\nsame post_ids as sampleSubmission:", set(test_df["post_id"]) == set(sample_sub["post_id"]))

train: (3360, 4) test: (1440, 4)
train rows without label: 0

label counts:
 label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64

words per post:
 count    3360.0
mean       16.5
std        25.0
min         1.0
25%         8.0
50%        13.0
75%        19.0
max       508.0
Name: n_words, dtype: float64

same post_ids as sampleSubmission: True


## 2. Feature Engineering Steps

In [3]:
### Add the code related to the feature engineering steps in cells inside this section

# TF-IDF features: word unigrams+bigrams plus character n-grams.
# Character n-grams help with short, informal posts (typos, slang, "sooo", emojis).
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

features = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
])

## 3. Model Implementation Steps

In [4]:
### Add the code related to the model implementation steps in cells inside this section

# Model 1: Linear SVM on TF-IDF. class_weight="balanced" because fear is rare and macro F1
# weights every class equally. 5-fold CV estimates the score without touching the test set.
import os
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

svm_model = Pipeline([("features", features), ("clf", LinearSVC(C=0.5, class_weight="balanced"))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(svm_model, train_df["text"], train_df["label"], cv=cv, scoring="f1_macro")
print("CV macro F1 per fold:", scores.round(3))
print(f"CV macro F1 mean: {scores.mean():.3f}")

# Train on all training posts and write a submission in sampleSubmission's order
svm_model.fit(train_df["text"], train_df["label"])
pred = pd.Series(svm_model.predict(test_df["text"]), index=test_df["post_id"])
submission = sample_sub[["post_id"]].copy()
submission["label"] = submission["post_id"].map(pred)

os.makedirs("submissions", exist_ok=True)
submission.to_csv("submissions/sub1_tfidf_svm.csv", index=False)
print(submission.shape, submission["label"].isna().sum(), "missing")
print(submission["label"].value_counts())

CV macro F1 per fold: [0.508 0.474 0.513 0.481 0.518]
CV macro F1 mean: 0.499
(1440, 2) 0 missing
label
joy         394
anger       326
sadness     269
surprise    178
disgust     165
fear        108
Name: count, dtype: int64


In [5]:
# Kaggle expects the ID column to be named "id", not "post_id" as in sampleSubmission.csv
submission = submission.rename(columns={"post_id": "id"})
submission.to_csv("submissions/sub1_tfidf_svm.csv", index=False)
print(submission.columns.tolist(), submission.shape)
print(submission.head(3))

['id', 'label'] (1440, 2)
        id    label
0  xyzx9jp    anger
1  l3vzzxt     fear
2  g680iug  sadness


In [6]:
# Check whether this machine has a CUDA GPU for fine-tuning, and which library versions we have
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch 2.14.1+cpu | transformers 5.17.0
CUDA available: False


In [7]:
# Model 2: fine-tune distilroberta-base (Apache 2.0 license) on the competition training posts only.
# No GPU on this machine, so: max length 128, 4 epochs, and one 15% validation split instead of 5-fold CV.
import os, time, copy
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(os.cpu_count())

MODEL_NAME = "distilroberta-base"
LABELS = sorted(train_df["label"].unique())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

# Hold out 15% of the training posts, keeping the class proportions, to pick the best epoch
tr, va = train_test_split(train_df, test_size=0.15, stratify=train_df["label"], random_state=42)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def make_loader(texts, labels=None, shuffle=False, batch_size=16):
    """Batches of tokenized posts; each batch is padded only to its own longest post."""
    if labels is None:
        labels = [-1] * len(texts)
    items = list(zip(texts, labels))
    def collate(batch):
        enc = tokenizer([b[0] for b in batch], padding=True, truncation=True,
                        max_length=128, return_tensors="pt")
        return dict(enc), torch.tensor([b[1] for b in batch])
    return DataLoader(items, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

train_loader = make_loader(tr["text"].tolist(), tr["label"].map(label2id).tolist(), shuffle=True)
val_loader = make_loader(va["text"].tolist(), va["label"].map(label2id).tolist())

# Class weights: rare classes (fear) count more in the loss, since macro F1 weighs all classes equally
counts = tr["label"].value_counts()
class_w = torch.tensor([len(tr) / (len(LABELS) * counts[l]) for l in LABELS], dtype=torch.float)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_w)

def predict(loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for inputs, _ in loader:
            preds.append(model(**inputs).logits.argmax(-1))
    return torch.cat(preds).numpy()

# Training: AdamW, 10% linear warm-up then linear decay, gradient clipping
EPOCHS = 4
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
warm = int(0.1 * total_steps)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min((s + 1) / warm, max(0.0, (total_steps - s) / (total_steps - warm))))

best_f1, best_state = -1.0, None
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    for step, (inputs, labels) in enumerate(train_loader):
        loss = loss_fn(model(**inputs).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        if step % 50 == 0:
            print(f"  epoch {epoch + 1} step {step}/{len(train_loader)} loss {loss.item():.3f}", flush=True)
    val_f1 = f1_score(va["label"].map(label2id), predict(val_loader), average="macro")
    print(f"epoch {epoch + 1}: validation macro F1 {val_f1:.3f} ({time.time() - t0:.0f}s)", flush=True)
    if val_f1 > best_f1:  # keep the best epoch, not the last one
        best_f1, best_state = val_f1, copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
print(f"best validation macro F1: {best_f1:.3f}")

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

C:\Users\USER\Desktop\DM2026-Lab2-Exercise\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1 step 0/179 loss 1.786
  epoch 1 step 50/179 loss 1.762
  epoch 1 step 100/179 loss 1.590
  epoch 1 step 150/179 loss 1.205
epoch 1: validation macro F1 0.562 (231s)
  epoch 2 step 0/179 loss 1.012
  epoch 2 step 50/179 loss 1.188
  epoch 2 step 100/179 loss 1.104
  epoch 2 step 150/179 loss 0.858
epoch 2: validation macro F1 0.573 (228s)
  epoch 3 step 0/179 loss 0.636
  epoch 3 step 50/179 loss 0.584
  epoch 3 step 100/179 loss 0.763
  epoch 3 step 150/179 loss 0.632
epoch 3: validation macro F1 0.581 (229s)
  epoch 4 step 0/179 loss 1.047
  epoch 4 step 50/179 loss 0.641
  epoch 4 step 100/179 loss 0.524
  epoch 4 step 150/179 loss 0.800
epoch 4: validation macro F1 0.606 (8542s)
best validation macro F1: 0.606


In [8]:
# Predict the 1440 test posts with the best epoch's model and write a Kaggle submission
test_loader = make_loader(test_df["text"].tolist())
test_pred = [id2label[i] for i in predict(test_loader)]

sub2 = pd.DataFrame({"id": test_df["post_id"], "label": test_pred})
sub2 = sub2.set_index("id").loc[sample_sub["post_id"]].reset_index()  # same order as sampleSubmission
sub2.columns = ["id", "label"]
sub2.to_csv("submissions/sub2_distilroberta.csv", index=False)
print(sub2.shape, sub2["label"].isna().sum(), "missing")
print(sub2["label"].value_counts())

# Save the fine-tuned model so we never have to retrain it just to predict again
model.save_pretrained("models/distilroberta_v1")
tokenizer.save_pretrained("models/distilroberta_v1")

(1440, 2) 0 missing
label
joy         355
anger       317
sadness     248
surprise    215
disgust     167
fear        138
Name: count, dtype: int64


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/distilroberta_v1\\tokenizer_config.json',
 'models/distilroberta_v1\\tokenizer.json')

In [9]:
# Model 3: same recipe with the larger roberta-base (MIT license), to test whether model size helps.
# The training loop is now a function, so later experiments change one argument instead of copying code.
import gc
del model          # free the distilroberta copy in memory (it is saved in models/distilroberta_v1)
gc.collect()

def batches(tok, texts, labels=None, shuffle=False, batch_size=16):
    """Batches of tokenized posts, each padded only to its own longest post."""
    if labels is None:
        labels = [-1] * len(texts)
    items = list(zip(texts, labels))
    def collate(batch):
        enc = tok([b[0] for b in batch], padding=True, truncation=True, max_length=128, return_tensors="pt")
        return dict(enc), torch.tensor([b[1] for b in batch])
    return DataLoader(items, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

def class_probs(mdl, loader):
    """Softmax probabilities for every post, kept so models can be averaged later (ensemble)."""
    mdl.eval()
    out = []
    with torch.no_grad():
        for inputs, _ in loader:
            out.append(torch.softmax(mdl(**inputs).logits, dim=-1))
    return torch.cat(out).numpy()

def fine_tune(model_name, epochs=4, lr=2e-5, seed=42):
    """Fine-tune on the 85% split, check the 15% validation split after each epoch, keep the best epoch."""
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    train_dl = batches(tok, tr["text"].tolist(), tr["label"].map(label2id).tolist(), shuffle=True)
    val_dl = batches(tok, va["text"].tolist())

    opt = torch.optim.AdamW(mdl.parameters(), lr=lr, weight_decay=0.01)
    total = epochs * len(train_dl)
    warm = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / (total - warm))))

    best_f1, best_state = -1.0, None
    for epoch in range(epochs):
        mdl.train()
        t0 = time.time()
        for step, (inputs, labels) in enumerate(train_dl):
            loss = loss_fn(mdl(**inputs).logits, labels)   # same class-weighted loss as Model 2
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad()
            if step % 50 == 0:
                print(f"  epoch {epoch + 1} step {step}/{len(train_dl)} loss {loss.item():.3f}", flush=True)
        val_f1 = f1_score(va["label"].map(label2id), class_probs(mdl, val_dl).argmax(1), average="macro")
        print(f"epoch {epoch + 1}: validation macro F1 {val_f1:.3f} ({time.time() - t0:.0f}s)", flush=True)
        if val_f1 > best_f1:
            best_f1, best_state = val_f1, copy.deepcopy(mdl.state_dict())

    mdl.load_state_dict(best_state)
    return mdl, tok, best_f1

rb_model, rb_tok, rb_f1 = fine_tune("roberta-base", epochs=4, lr=2e-5)
print(f"roberta-base best validation macro F1: {rb_f1:.3f}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

C:\Users\USER\Desktop\DM2026-Lab2-Exercise\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1 step 0/179 loss 1.796
  epoch 1 step 50/179 loss 1.783
  epoch 1 step 100/179 loss 1.737
  epoch 1 step 150/179 loss 1.193
epoch 1: validation macro F1 0.566 (536s)
  epoch 2 step 0/179 loss 1.290
  epoch 2 step 50/179 loss 1.345
  epoch 2 step 100/179 loss 0.797
  epoch 2 step 150/179 loss 1.087
epoch 2: validation macro F1 0.628 (583s)
  epoch 3 step 0/179 loss 1.030
  epoch 3 step 50/179 loss 0.758
  epoch 3 step 100/179 loss 0.893
  epoch 3 step 150/179 loss 0.652
epoch 3: validation macro F1 0.621 (593s)
  epoch 4 step 0/179 loss 0.326
  epoch 4 step 50/179 loss 0.623
  epoch 4 step 100/179 loss 0.386
  epoch 4 step 150/179 loss 0.748
epoch 4: validation macro F1 0.615 (588s)
roberta-base best validation macro F1: 0.628


In [10]:
# Predict the test posts with roberta-base's best epoch and write submission 3.
# Also save the class probabilities for validation and test, so models can be averaged later.
test_dl_rb = batches(rb_tok, test_df["text"].tolist())
val_dl_rb = batches(rb_tok, va["text"].tolist())
rb_test_probs = class_probs(rb_model, test_dl_rb)
rb_val_probs = class_probs(rb_model, val_dl_rb)

os.makedirs("preds", exist_ok=True)
np.save("preds/roberta_test_probs.npy", rb_test_probs)
np.save("preds/roberta_val_probs.npy", rb_val_probs)

sub3 = pd.DataFrame({"id": test_df["post_id"], "label": [id2label[i] for i in rb_test_probs.argmax(1)]})
sub3 = sub3.set_index("id").loc[sample_sub["post_id"]].reset_index()  # same order as sampleSubmission
sub3.columns = ["id", "label"]
sub3.to_csv("submissions/sub3_roberta_base.csv", index=False)
print(sub3.shape, sub3["label"].isna().sum(), "missing")
print(sub3["label"].value_counts())

rb_model.save_pretrained("models/roberta_base_v1")
rb_tok.save_pretrained("models/roberta_base_v1")

(1440, 2) 0 missing
label
joy         415
anger       322
sadness     315
disgust     155
surprise    121
fear        112
Name: count, dtype: int64


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/roberta_base_v1\\tokenizer_config.json',
 'models/roberta_base_v1\\tokenizer.json')

In [11]:
# Ensemble: average the class probabilities of distilroberta (saved in models/) and roberta-base.
# Check on the validation split first; only submit if the average beats roberta-base alone.
db_model = AutoModelForSequenceClassification.from_pretrained("models/distilroberta_v1")
db_tok = AutoTokenizer.from_pretrained("models/distilroberta_v1")
db_val_probs = class_probs(db_model, batches(db_tok, va["text"].tolist()))
db_test_probs = class_probs(db_model, batches(db_tok, test_df["text"].tolist()))
np.save("preds/distilroberta_val_probs.npy", db_val_probs)
np.save("preds/distilroberta_test_probs.npy", db_test_probs)

y_val = va["label"].map(label2id).to_numpy()
for name, probs in [("distilroberta", db_val_probs), ("roberta-base", rb_val_probs),
                    ("average of both", (db_val_probs + rb_val_probs) / 2)]:
    print(f"{name:16s} validation macro F1: {f1_score(y_val, probs.argmax(1), average='macro'):.3f}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

distilroberta    validation macro F1: 0.606
roberta-base     validation macro F1: 0.628
average of both  validation macro F1: 0.605


In [12]:
# Model 4: roberta-base retrained on ALL 3360 training posts for 2 epochs
# (epoch 2 was the best epoch on the validation split above). No validation split is left,
# so the Kaggle public score is the only check.
del db_model
gc.collect()

def fine_tune_full(model_name, epochs, lr=2e-5, seed=42):
    """Same recipe as fine_tune, but on every training post and for a fixed number of epochs."""
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    train_dl = batches(tok, train_df["text"].tolist(), train_df["label"].map(label2id).tolist(), shuffle=True)

    opt = torch.optim.AdamW(mdl.parameters(), lr=lr, weight_decay=0.01)
    total = epochs * len(train_dl)
    warm = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / (total - warm))))

    for epoch in range(epochs):
        mdl.train()
        t0 = time.time()
        for step, (inputs, labels) in enumerate(train_dl):
            loss = loss_fn(mdl(**inputs).logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad()
            if step % 50 == 0:
                print(f"  epoch {epoch + 1} step {step}/{len(train_dl)} loss {loss.item():.3f}", flush=True)
        print(f"epoch {epoch + 1} done ({time.time() - t0:.0f}s)", flush=True)
    return mdl, tok

full_model, full_tok = fine_tune_full("roberta-base", epochs=2)

full_test_probs = class_probs(full_model, batches(full_tok, test_df["text"].tolist()))
np.save("preds/roberta_full_test_probs.npy", full_test_probs)

sub4 = pd.DataFrame({"id": test_df["post_id"], "label": [id2label[i] for i in full_test_probs.argmax(1)]})
sub4 = sub4.set_index("id").loc[sample_sub["post_id"]].reset_index()
sub4.columns = ["id", "label"]
sub4.to_csv("submissions/sub4_roberta_full.csv", index=False)
print(sub4.shape, sub4["label"].isna().sum(), "missing")
print(sub4["label"].value_counts())

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1 step 0/210 loss 1.845
  epoch 1 step 50/210 loss 1.751
  epoch 1 step 100/210 loss 1.369
  epoch 1 step 150/210 loss 1.149
  epoch 1 step 200/210 loss 1.053
epoch 1 done (667s)
  epoch 2 step 0/210 loss 1.087
  epoch 2 step 50/210 loss 1.009
  epoch 2 step 100/210 loss 1.078
  epoch 2 step 150/210 loss 0.977
  epoch 2 step 200/210 loss 0.963
epoch 2 done (685s)
(1440, 2) 0 missing
label
joy         354
anger       303
sadness     265
surprise    202
disgust     189
fear        127
Name: count, dtype: int64


### Kaggle GPU experiments

The cells below were run on a **Kaggle notebook with a GPU (Tesla T4)**, because fine-tuning on this laptop's CPU took 4 to 10 minutes per epoch.
They read the competition files from `/kaggle/input/...` and need a CUDA GPU, so they are **not re-run here**.
Each cell's results are written in its comments and summarized in the report above.

Main steps:
1. Same preprocessing and recipe as the local cells (class-weighted loss, max length 128), plus fp16 mixed precision for speed.
2. Single 85/15 split experiments: roberta-large, deberta-v3-base, and seed ensembles.
3. **5-fold cross-validation** of roberta-base and roberta-large, so every training post is scored once (out-of-fold, 3,360 posts) instead of only 504.
4. Final submissions: averages of roberta-large and roberta-base predictions.

In [ ]:
# [Kaggle GPU] Cell K1: load the competition files and define the helpers used by every experiment.
# Same data, same 85/15 stratified split (random_state=42) and class-weighted loss as the laptop cells.
import json, time
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DATA_DIR = "/kaggle/input/competitions/2026-dm-lab-2-private-competition"
device = torch.device("cuda")

with open(f"{DATA_DIR}/final_posts.json", encoding="utf-8") as f:
    posts = pd.DataFrame(json.load(f))
df = posts.merge(pd.read_csv(f"{DATA_DIR}/data_identification.csv"), on="post_id", how="left")
df = df.merge(pd.read_csv(f"{DATA_DIR}/emotion.csv"), on="post_id", how="left")
sample_sub = pd.read_csv(f"{DATA_DIR}/sampleSubmission.csv")
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = df[df["set"] == "test"].reset_index(drop=True)

LABELS = sorted(train_df["label"].unique())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

tr, va = train_test_split(train_df, test_size=0.15, stratify=train_df["label"], random_state=42)
y_val = va["label"].map(label2id).to_numpy()

counts = tr["label"].value_counts()
class_w = torch.tensor([len(tr) / (len(LABELS) * counts[l]) for l in LABELS], dtype=torch.float, device=device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_w)
print("train", train_df.shape, "test", test_df.shape, "| split", len(tr), len(va))
# Output on Kaggle: train (3360, 4) test (1440, 4) | split 2856 504

def batches(tok, texts, labels=None, shuffle=False, batch_size=16):
    """Batches of tokenized posts (max 128 tokens), each padded only to its own longest post."""
    if labels is None:
        labels = [-1] * len(texts)
    items = list(zip(texts, labels))
    def collate(batch):
        enc = tok([b[0] for b in batch], padding=True, truncation=True, max_length=128, return_tensors="pt")
        return dict(enc), torch.tensor([b[1] for b in batch])
    return DataLoader(items, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

def class_probs(mdl, loader):
    """Softmax probabilities for every post, kept so models can be averaged (ensembled)."""
    mdl.eval()
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for inputs, _ in loader:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            out.append(torch.softmax(mdl(**inputs).logits.float(), dim=-1).cpu())
    return torch.cat(out).numpy()

In [ ]:
# [Kaggle GPU] Cell K2: the training function (85/15 split) and the submission writer.
def fine_tune(model_name, epochs=4, lr=2e-5, batch_size=16, seed=42):
    """Train on the 85% split; after each epoch score the 15% split and keep the best epoch's
    validation and test probabilities. Returns (best_f1, val_probs, test_probs)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    # .float(): deberta-v3 checkpoints load as fp16, and mixed precision needs fp32 weights
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id).float().to(device)
    train_dl = batches(tok, tr["text"].tolist(), tr["label"].map(label2id).tolist(), shuffle=True, batch_size=batch_size)
    val_dl = batches(tok, va["text"].tolist(), batch_size=64)
    test_dl = batches(tok, test_df["text"].tolist(), batch_size=64)

    # AdamW, 10% linear warm-up then linear decay, gradient clipping, fp16 mixed precision
    opt = torch.optim.AdamW(mdl.parameters(), lr=lr, weight_decay=0.01)
    total = epochs * len(train_dl)
    warm = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / (total - warm))))
    scaler = torch.amp.GradScaler("cuda")

    best_f1, best_val, best_test = -1.0, None, None
    for epoch in range(epochs):
        mdl.train()
        t0 = time.time()
        for inputs, labels in train_dl:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = mdl(**inputs).logits
            loss = loss_fn(logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()
        val_probs = class_probs(mdl, val_dl)
        val_f1 = f1_score(y_val, val_probs.argmax(1), average="macro")
        print(f"epoch {epoch + 1}: validation macro F1 {val_f1:.3f} ({time.time() - t0:.0f}s)", flush=True)
        if val_f1 > best_f1:   # keep the best epoch, not the last one
            best_f1, best_val, best_test = val_f1, val_probs, class_probs(mdl, test_dl)

    del mdl
    torch.cuda.empty_cache()
    return best_f1, best_val, best_test

def make_submission(test_probs, name):
    """Write /kaggle/working/<name>.csv with columns id,label in sampleSubmission's

In [ ]:
# [Kaggle GPU] Cell K3: experiments on the 85/15 split, then seed ensembles.
# Larger models get a smaller learning rate. Results (best validation macro F1 on the 504 posts):
#   roberta-large (5 epochs, lr 1e-5): 0.626   -> sub5, not submitted (no better than roberta-base 0.628)
#   deberta-v3-base (5 epochs, lr 2e-5): 0.586 -> sub6, not submitted; averaged with roberta-large: 0.611
f1_large, val_large, test_large = fine_tune("roberta-large", epochs=5, lr=1e-5)
f1_deb, val_deb, test_deb = fine_tune("microsoft/deberta-v3-base", epochs=5, lr=2e-5)

# Seed ensembles: the same model trained with different random seeds makes different mistakes,
# so averaging the probabilities is steadier than any single run.
#   roberta-large seeds 42 / 7 / 2026 / 1 / 123: 0.626 / 0.622 / 0.626 / 0.614 / 0.627
#   roberta-base  seeds 42 / 7 / 2026:           0.617 / 0.604 / 0.612
# The roberta-base scores alone (0.604 to 0.617) show that 0.01 on 504 posts is mostly noise.
runs = {}
for name, lr, epochs, seeds in [("roberta-large", 1e-5, 5, [42, 7, 2026, 1, 123]),
                                ("roberta-base", 2e-5, 4, [42, 7, 2026])]:
    for seed in seeds:
        f1_s, val_s, test_s = fine_tune(name, epochs=epochs, lr=lr, seed=seed)
        runs[(name, seed)] = (val_s, test_s)

def avg(model_names, part, seeds=None):
    """Average the val (part=0) or test (part=1) probabilities of the chosen runs."""
    return np.mean([p[part] for (n, s), p in runs.items()
                    if n in model_names and (seeds is None or s in seeds)], axis=0)

# Validation macro F1 of each average and its Kaggle public score:
#   roberta-base x3 + roberta-large x1                  val 0.635 -> sub7,  public 0.5928
#   roberta-large x3 (seeds 42, 7, 2026)                val 0.637 -> sub8,  public 0.5929
#   roberta-large x3 + roberta-base x3                  val 0.635 -> sub9,  public 0.5996  (selected)
#   roberta-large x5 + roberta-base x3                  val 0.637 -> sub10, public 0.5967
#   roberta-large x5                                    val 0.637 -> sub11, public 0.5855
# Both times, adding roberta-base raised the public score (sub8 -> sub9, sub11 -> sub10).
make_submission(avg(["roberta-large"], 1, seeds=[42, 7, 2026]), "sub8_roberta_large_x3")
make_submission(avg(["roberta-large", "roberta-base"], 1, seeds=[42, 7, 2026]), "sub9_all6")
make_submission(avg(["roberta-large", "roberta-base"], 1), "sub10_large5_base3")
make_submission(avg(["roberta-large"], 1), "sub11_large5")

In [ ]:
# [Kaggle GPU] Cell K4: 5-fold cross-validation, the final approach.
# Every training post is used for training in 4 folds and scored once, so the out-of-fold (OOF)
# score covers all 3360 posts instead of 504. Epochs are fixed per model (no best-epoch picking),
# so the OOF score is not tuned on its own folds. Test predictions are the average of the 5 fold models.
from sklearn.model_selection import StratifiedKFold

y_all = train_df["label"].map(label2id).to_numpy()
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(train_df["text"], y_all))

def train_fold(model_name, tr_idx, va_idx, epochs, lr, seed=42):
    """Train on 4 folds, return probabilities for the held-out fold and for the test posts."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id).float().to(device)
    tr_f, va_f = train_df.iloc[tr_idx], train_df.iloc[va_idx]

    counts = tr_f["label"].value_counts()   # class weights from this fold's training part
    w = torch.tensor([len(tr_f) / (len(LABELS) * counts[l]) for l in LABELS], dtype=torch.float, device=device)
    fold_loss = torch.nn.CrossEntropyLoss(weight=w)

    train_dl = batches(tok, tr_f["text"].tolist(), tr_f["label"].map(label2id).tolist(), shuffle=True)
    opt = torch.optim.AdamW(mdl.parameters(), lr=lr, weight_decay=0.01)
    total = epochs * len(train_dl)
    warm = int(0.1 * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / (total - warm))))
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(epochs):
        mdl.train()
        for inputs, labels in train_dl:
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = mdl(**inputs).logits
            loss = fold_loss(logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()

    va_probs = class_probs(mdl, batches(tok, va_f["text"].tolist(), batch_size=64))
    te_probs = class_probs(mdl, batches(tok, test_df["text"].tolist(), batch_size=64))
    del mdl
    torch.cuda.empty_cache()
    return va_probs, te_probs

def run_cv(model_name, epochs, lr, seed=42):
    """5 folds -> out-of-fold probabilities for all training posts + averaged test probabilities."""
    oof = np.zeros((len(train_df), len(LABELS)))
    test_avg = np.zeros((len(test_df), len(LABELS)))
    for k, (tr_idx, va_idx) in enumerate(folds):
        va_p, te_p = train_fold(model_name, tr_idx, va_idx, epochs, lr, seed)
        oof[va_idx] = va_p
        test_avg += te_p / len(folds)
        print(f"{model_name} fold {k + 1}: macro F1 {f1_score(y_all[va_idx], va_p.argmax(1), average='macro'):.3f}")
    print(f"==> {model_name} OOF macro F1: {f1_score(y_all, oof.argmax(1), average='macro'):.3f}")
    return oof, test_avg

# roberta-base:  folds 0.576 / 0.565 / 0.617 / 0.615 / 0.589 -> OOF 0.593 -> sub12, not submitted
# roberta-large: folds 0.566 / 0.585 / 0.617 / 0.611 / 0.576 -> OOF 0.591 -> sub13, not submitted
# On all 3360 posts the two models are equally good alone; the earlier 0.626 vs 0.628 gap was noise.
oof_base, test_base = run_cv("roberta-base", epochs=4, lr=2e-5)
oof_large, test_large_cv = run_cv("roberta-large", epochs=4, lr=1e-5)
make_submission(test_base, "sub12_cv_roberta_base")
make_submission(test_large_cv, "sub13_cv_roberta_large")

# Final model: average of the two -> OOF 0.602 (+0.01 over either alone) -> sub14, public 0.5965 (selected)
oof_ens = (oof_large + oof_base) / 2
test_ens = (test_large_cv + test_base) / 2
print(f"large + base OOF macro F1: {f1_score(y_all, oof_ens.argmax(1), average='macro'):.3f}")
make_submission(test_ens, "sub14_cv_large_base")

In [ ]:
# [Kaggle GPU] Cell K5: experiments that did not help, and the error analysis of the final model.

# 1) ELECTRA-large, 5-fold (lr 1e-5, 4 epochs): folds 0.439 / 0.538 / 0.069 / 0.444 / 0.054 -> OOF 0.361.
#    Two folds collapsed to predicting one class (a known instability of ELECTRA-large on small data).
#    Adding it to the ensemble: 0.602 -> 0.601. Not used (sub16, sub17 not submitted).
oof_electra, test_electra = run_cv("google/electra-large-discriminator", epochs=4, lr=1e-5)

# 2) deberta-v3-large, checked on fold 1 only (5 epochs, lr 1e-5): 0.560, vs roberta-large 0.566
#    and roberta-base 0.576 on the same fold. In the ensemble on fold 1: 0.588 -> 0.591 (about 2 posts),
#    so the other 4 folds (about 30 minutes of GPU) were not run.
tr_idx, va_idx = folds[0]
va_p, te_p = train_fold("microsoft/deberta-v3-large", tr_idx, va_idx, epochs=5, lr=1e-5)

# 3) Per-class weights on the probabilities, tuned for macro F1. Honest check: tune on 4 folds' OOF,
#    score the 5th. Result 0.602 -> 0.604 (1 or 2 posts), with large weights such as anger x1.4,
#    which looks like fitting noise. Not used (sub15 not submitted).
def tune_weights(probs, y, grid=np.arange(0.6, 1.65, 0.05), passes=3):
    w = np.ones(probs.shape[1])
    for _ in range(passes):
        for c in range(probs.shape[1]):
            best_g, best_f = w[c], -1.0
            for g in grid:
                w[c] = g
                f = f1_score(y, (probs * w).argmax(1), average="macro")
                if f > best_f:
                    best_f, best_g = f, g
            w[c] = best_g
    return w

honest_pred = np.zeros(len(y_all), dtype=int)
for tr_idx, va_idx in folds:
    honest_pred[va_idx] = (oof_ens[va_idx] * tune_weights(oof_ens[tr_idx], y_all[tr_idx])).argmax(1)
print(f"tuned weights, honest check: {f1_score(y_all, honest_pred, average='macro'):.3f}")

# 4) Error analysis of the final ensemble (OOF, 3360 posts): accuracy 0.623, macro F1 0.602.
#    F1 per class: joy 0.759, sadness 0.624, fear 0.611, surprise 0.598, anger 0.590, disgust 0.432.
#    Most confused pairs (both directions): anger <-> disgust 199, anger <-> sadness 127,
#    joy <-> surprise 118, disgust <-> sadness 104. Disgust is the bottleneck: about a quarter of
#    disgust posts are predicted as anger, two emotions that both express disapproval.
from sklearn.metrics import classification_report, confusion_matrix
pred = oof_ens.argmax(1)
print(classification_report(y_all, pred, target_names=LABELS, digits=3))
print(pd.Da